# 07 — Structured Data

Pre-filling assistant message + stop sequences → чистий JSON без
markdown-обгортки й пояснень. Приклад курсу: **EventBridge Rule Generator**.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.append("..")
from helpers.chat_utils import add_user_message, add_assistant_message, chat

print("Helpers завантажено (chat тепер приймає stop_sequences=... і output_config=...)")

## ⚠️ Важливо: prefill НЕ працює з Claude Sonnet 5

Claude Sonnet 5 (і будь-яка модель новіша за 4.5) **повністю прибрала**
підтримку assistant message prefill — API вимагає, щоб розмова
**завжди закінчувалась user message**. Спроба підставити pre-filled
assistant message кидає:
```
400: This model does not support assistant message prefill.
The conversation must end with a user message.
```

**Два виходи:**
- **Обхід А** (як у курсі) — `model_override` на модель, що ще підтримує prefill (`claude-sonnet-4-5-20250929`)
- **Обхід Б** (сучасний, рекомендований Anthropic) — `output_config` замість prefill+stop, працює прямо на Sonnet 5

## Без техніки — типова "брудна" відповідь

Побач, як Claude сам обгортає JSON у markdown-код і додає пояснення.

In [ ]:
messages_plain = []
add_user_message(messages_plain, "Generate a very short event bridge rule as json")

print(chat(messages_plain))

## Обхід А — prefill + stop sequence (як у курсі, на старшій моделі)

1. Просимо JSON
2. Pre-fill відповіді асистента символом ` ```json ` — Claude "думає",
   що вже почав code block
3. `stop_sequences=["```"]` — зупиняємо генерацію на закриваючому символі
4. `model_override` — бо prefill не підтримується на дефолтній Sonnet 5

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # pre-fill

text = chat(
    messages,
    stop_sequences=["```"],
    model_override="claude-sonnet-4-5-20250929",  # тут prefill ще підтримується
)
print(repr(text))  # repr — щоб побачити \n символи, про які попереджає курс

## Обробка відповіді — парсимо як реальний JSON

In [ ]:
import json

clean_json = json.loads(text.strip())  # .strip() прибирає зайві \n з початку/кінця
print(clean_json)
print(type(clean_json))

## ➕ Обхід Б — `output_config` (сучасний, працює на Sonnet 5 напряму)

Офіційна заміна Anthropic для prefill-техніки: `output_config.format`
з JSON schema — гарантує (schema-validated!) валідний JSON, без
необхідності "обманювати" модель pre-fill'ом. Розмова закінчується
звичайним user message — жодних 400-помилок.

In [ ]:
# ⚠️ ВАЖЛИВО: output_config вимагає "additionalProperties": false
# на КОЖНОМУ object-рівні схеми (і верхньому, і вкладених!) — інакше 400:
# 'For "object" type, "additionalProperties" must be explicitly set to false'
output_config = {
    "format": {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "source": {"type": "array", "items": {"type": "string"}},
                "detail-type": {"type": "array", "items": {"type": "string"}},
                "detail": {
                    "type": "object",
                    "properties": {
                        "state": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["state"],
                    "additionalProperties": False,  # вкладений object — теж треба!
                },
            },
            "required": ["source", "detail-type", "detail"],
            "additionalProperties": False,  # верхній рівень
        },
    }
}

messages_modern = []
add_user_message(messages_modern, "Generate a very short event bridge rule as json")

text_modern = chat(messages_modern, output_config=output_config)  # без prefill, без stop_sequences
print(text_modern)

clean_json_modern = json.loads(text_modern)
print("\nParsed:", clean_json_modern)

## 🧪 Своя перевірка

Та сама техніка (Обхід А) працює і для Python-коду (не тільки JSON) —
заміни pre-fill на ` ```python ` замість ` ```json `.

In [ ]:
my_messages = []
add_user_message(my_messages, "Write a Python function that reverses a string")
add_assistant_message(my_messages, "```python")

code = chat(my_messages, stop_sequences=["```"], model_override="claude-sonnet-4-5-20250929")
print(code)

---

## ➕ Structured Data Exercise (з відео)

**Завдання:** використовуючи ТІЛЬКИ prefill + stop sequences,
отримати 3 різні AWS CLI команди в ОДНІЙ відповіді, без жодних
коментарів чи пояснень.

### ❌ Пастка — наївний prefill

Якщо просто підставити `"Here are all three commands ```bash"`,
Claude сам вирішує обгорнути КОЖНУ команду в ОКРЕМИЙ code block.
`stop_sequences=["```"]` тоді спрацьовує на ПЕРШОМУ закриваючому
` ``` ` — обрізає результат до однієї команди. Спробуй сам, побач
цю поведінку наживо:

In [ ]:
messages_naive = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages_naive, prompt)
add_assistant_message(messages_naive, "Here are all three commands ```bash")

text_naive = chat(
    messages_naive,
    stop_sequences=["```"],
    model_override="claude-sonnet-4-5-20250929",  # prefill не працює на Sonnet 5
)
print(repr(text_naive.strip()))  # ⚠️ ймовірно лише ОДНА команда, не три

### ✅ Рішення — детальніший prefill

**Ключовий інсайт з підказки відео: "message prefilling isn't
limited to just characters like ` ``` `"** — pre-fill може бути
повноцінною інструкцією, не тільки коротким delimiter'ом. Claude
"думає", що сам це вже написав — і дотримується рамки сильніше,
ніж якби те саме сказали через system prompt.

In [ ]:
messages_fixed = []
add_user_message(messages_fixed, prompt)
add_assistant_message(
    messages_fixed,
    "Here are all three commands in a single block without any comments:\n```bash",
)

text_fixed = chat(
    messages_fixed,
    stop_sequences=["```"],
    model_override="claude-sonnet-4-5-20250929",
)
print(text_fixed.strip())  # тепер очікуємо усі 3 команди, кожна на своєму рядку